Import Library

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelBinarizer
from tensorflow.keras.utils import to_categorical
import cv2

Setup Untuk GPU

In [4]:
#gpus = tf.config.list_physical_devices('GPU')
#if gpus:
    #tf.config.experimental.set_memory_growth(gpus[0], True)
    #print("✅ GPU aktif dan digunakan untuk training:", gpus[0])
#else:
print("⚠️ GPU tidak terdeteksi, training berjalan di CPU.")

⚠️ GPU tidak terdeteksi, training berjalan di CPU.


Memasukkan Dataset dan konfigurasi dasar

In [5]:
data_dir = r"C:\Users\Lenovo\Klasifikasi_ikan\datasets\normalized_output"
classes = ["segar", "tidak_segar"]
img_height, img_width = 224, 224
batch_size = 16
epochs = 30
k_folds = 5

#  Memuat dataset 
print("📥 Memuat dataset...")
data, labels = [], []

for label in classes:
    path = os.path.join(data_dir, label)
    for img_name in os.listdir(path):
        img_path = os.path.join(path, img_name)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.resize(img, (img_width, img_height))
            data.append(img)
            labels.append(label)

data = np.array(data, dtype="float32") / 255.0
labels = np.array(labels)

print(f"✅ Total data: {len(data)} gambar")

📥 Memuat dataset...
✅ Total data: 1000 gambar


Memasukkan indeks K-Fold

In [6]:
# Encoding label 
lb = LabelBinarizer()
labels = lb.fit_transform(labels)
labels = to_categorical(labels)
num_classes = len(lb.classes_)

# Setup K-FOLD
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

acc_per_fold = []
loss_per_fold = []

# Folder untuk menyimpan model
os.makedirs("models", exist_ok=True)

# Untuk menyimpan data terakhir (untuk confusion matrix)
X_val_final, y_val_final, y_pred_final = None, None, None

Proses training

In [7]:
fold_no = 1
for train_idx, val_idx in kf.split(data, labels):
    print(f"\n==============================")
    print(f"🔹 Fold {fold_no}/{k_folds} sedang dilatih...")
    print(f"==============================")

    X_train, X_val = data[train_idx], data[val_idx]
    y_train, y_val = labels[train_idx], labels[val_idx]

    # Model dasar MobileNetV2
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    predictions = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=predictions)

    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    checkpoint_path = f"models/model_fold_{fold_no}.keras"
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ModelCheckpoint(filepath=checkpoint_path, save_best_only=True, monitor='val_accuracy', mode='max')
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1
    )

    # Evaluasi per fold
    scores = model.evaluate(X_val, y_val, verbose=0)
    print(f"✅ Fold {fold_no} - Akurasi: {scores[1]*100:.2f}% | Loss: {scores[0]:.4f}")

    acc_per_fold.append(scores[1] * 100)
    loss_per_fold.append(scores[0])

    # Simpan fold terakhir untuk evaluasi confusion matrix
    if fold_no == k_folds:
        y_pred_final = model.predict(X_val)
        X_val_final = X_val
        y_val_final = y_val

    fold_no += 1

# === 6. Hasil akhir ===
print("\n==============================")
print("📊 Hasil K-Fold Validation")
print("==============================")
for i in range(k_folds):
    print(f"Fold {i+1}: Akurasi = {acc_per_fold[i]:.2f}%, Loss = {loss_per_fold[i]:.4f}")

print(f"\n🔹 Rata-rata Akurasi: {np.mean(acc_per_fold):.2f}%")
print(f"🔹 Rata-rata Loss: {np.mean(loss_per_fold):.4f}")

# === 7. Simpan hasil untuk evaluasi confusion matrix ===
np.savez("model_evaluation_data.npz",
         X_val_final=X_val_final,
         y_val_final=y_val_final,
         y_pred_final=y_pred_final)

print("\n✅ Data evaluasi (X_val, y_val, prediksi) telah disimpan ke 'model_evaluation_data.npz'")


🔹 Fold 1/5 sedang dilatih...


Epoch 1/30


50/50 [==============================] - 7s 108ms/step - loss: 0.9877 - accuracy: 0.4863 - val_loss: 0.6977 - val_accuracy: 0.6000
Epoch 2/30
50/50 [==============================] - 5s 93ms/step - loss: 0.7540 - accuracy: 0.5788 - val_loss: 0.5620 - val_accuracy: 0.7250
Epoch 3/30
50/50 [==============================] - 5s 94ms/step - loss: 0.6488 - accuracy: 0.6562 - val_loss: 0.4765 - val_accuracy: 0.7800
Epoch 4/30
50/50 [==============================] - 5s 95ms/step - loss: 0.5922 - accuracy: 0.7138 - val_loss: 0.4086 - val_accuracy: 0.8650
Epoch 5/30
50/50 [==============================] - 4s 89ms/step - loss: 0.5071 - accuracy: 0.7600 - val_loss: 0.3708 - val_accuracy: 0.8600
Epoch 6/30
50/50 [==============================] - 5s 91ms/step - loss: 0.4726 - accuracy: 0.7862 - val_loss: 0.3410 - val_accuracy: 0.8850
Epoch 7/30
50/50 [==============================] - 5s 92ms/step - loss: 0.4589 - accuracy: 0.7962 - val_loss: 0.3207 -

Percobaan training ulang

In [6]:
import os
import numpy as np
import cv2
from tensorflow.keras.utils import to_categorical

# === Folder Dataset ===
path_segar = r"C:\Users\Lenovo\Klasifikasi_ikan\datasets\normalized_output\segar"
path_tidak_segar = r"C:\Users\Lenovo\Klasifikasi_ikan\datasets\normalized_output\tidak_segar"

img_height, img_width = 224, 224

# === Fungsi untuk membaca dan resize citra ===
def load_images_from_folder(folder, label):
    images = []
    labels = []
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.resize(img, (img_width, img_height))
            img = img / 255.0  # Normalisasi [0–1]
            images.append(img)
            labels.append(label)
    return images, labels

# === Muat semua citra ===
images_segar, labels_segar = load_images_from_folder(path_segar, 0)  # Label 0 = Segar
images_tdk_segar, labels_tdk_segar = load_images_from_folder(path_tidak_segar, 1)  # Label 1 = Tidak Segar

# === Gabungkan data ===
X_data = np.array(images_segar + images_tdk_segar, dtype="float32")
y_data = np.array(labels_segar + labels_tdk_segar)

# === One-hot encoding label ===
y_data = to_categorical(y_data, num_classes=2)

print(f"✅ Dataset siap digunakan")
print(f"Total data: {len(X_data)}")
print(f"Shape X_data: {X_data.shape}")
print(f"Shape y_data: {y_data.shape}")


✅ Dataset siap digunakan
Total data: 1000
Shape X_data: (1000, 224, 224, 3)
Shape y_data: (1000, 2)


In [9]:
# === 1. Import Library ===
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# === 2. Parameter Dasar ===
img_height, img_width = 224, 224
num_classes = 2
k_folds = 5
epochs = 30
batch_size = 16
learning_rate = 0.0001

# === 3. Setup Folder Hasil ===
base_dir = r"C:\Users\Lenovo\Klasifikasi_ikan\hasil_model"

folders = [
    os.path.join(base_dir, "base", "models"),
    os.path.join(base_dir, "base", "history"),
    os.path.join(base_dir, "base", "plots"),
    os.path.join(base_dir, "base", "evaluation"),
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("[INFO] Struktur folder hasil_model berhasil dibuat!")

# === 4. Dataset ===
# Pastikan X_data dan y_data sudah ada sebelumnya
# Contoh: X_data.shape = (1000, 224, 224, 3), y_data.shape = (1000, 2)
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

acc_per_fold = []
loss_per_fold = []

# === 5. Training per Fold ===
fold_no = 1
for train_index, val_index in kf.split(X_data, y_data):
    print(f"\n==============================")
    print(f"🔹 Fold {fold_no}/{k_folds} sedang dilatih...")
    print(f"==============================")

    X_train, X_val = X_data[train_index], X_data[val_index]
    y_train, y_val = y_data[train_index], y_data[val_index]

    # === Model MobileNetV2 ===
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    predictions = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=predictions)

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # === Path untuk setiap hasil per fold ===
    checkpoint_path = os.path.join(base_dir, "base", "models", f"model_fold_{fold_no}.keras")
    csv_path = os.path.join(base_dir, "base", "history", f"history_fold_{fold_no}.csv")
    plot_acc_path = os.path.join(base_dir, "base", "plots", f"accuracy_fold_{fold_no}.png")
    plot_loss_path = os.path.join(base_dir, "base", "plots", f"loss_fold_{fold_no}.png")

    # === Callback ===
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ModelCheckpoint(filepath=checkpoint_path, save_best_only=True, monitor='val_accuracy', mode='max')
    ]

    # === Training ===
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1
    )

    # === Simpan history ke CSV ===
    pd.DataFrame(history.history).to_csv(csv_path, index=False)

    # === Plot Accuracy ===
    plt.figure(figsize=(8,4))
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'Akurasi Fold {fold_no}')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.savefig(plot_acc_path)
    plt.close()

    # === Plot Loss ===
    plt.figure(figsize=(8,4))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'Loss Fold {fold_no}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(plot_loss_path)
    plt.close()

    # === Evaluasi ===
    scores = model.evaluate(X_val, y_val, verbose=0)
    print(f"✅ Fold {fold_no} - Akurasi: {scores[1]*100:.2f}% | Loss: {scores[0]:.4f}")

    acc_per_fold.append(scores[1] * 100)
    loss_per_fold.append(scores[0])

    # Simpan data evaluasi terakhir
    if fold_no == k_folds:
        y_pred_final = model.predict(X_val)
        X_val_final = X_val
        y_val_final = y_val

    fold_no += 1

# === 6. Ringkasan Akhir ===
print("\n==============================")
print("📊 Hasil K-Fold Validation")
print("==============================")
for i in range(k_folds):
    print(f"Fold {i+1}: Akurasi = {acc_per_fold[i]:.2f}%, Loss = {loss_per_fold[i]:.4f}")

mean_acc = np.mean(acc_per_fold)
mean_loss = np.mean(loss_per_fold)
print(f"\n🔹 Rata-rata Akurasi: {mean_acc:.2f}%")
print(f"🔹 Rata-rata Loss: {mean_loss:.4f}")

# === 7. Simpan Data Evaluasi & Ringkasan ===
np.savez(os.path.join(base_dir, "base", "evaluation", "model_evaluation_data.npz"),
         X_val_final=X_val_final,
         y_val_final=y_val_final,
         y_pred_final=y_pred_final)

with open(os.path.join(base_dir, "base", "evaluation", "kfold_summary.txt"), "w") as f:
    for i in range(k_folds):
        f.write(f"Fold {i+1}: Akurasi = {acc_per_fold[i]:.2f}%, Loss = {loss_per_fold[i]:.4f}\n")
    f.write(f"\nRata-rata Akurasi: {mean_acc:.2f}%\n")
    f.write(f"Rata-rata Loss: {mean_loss:.4f}\n")

print("\n✅ Semua hasil training dan evaluasi tersimpan di folder:")
print(base_dir)



[INFO] Struktur folder hasil_model berhasil dibuat!

🔹 Fold 1/5 sedang dilatih...
Epoch 1/30
50/50 [==============================] - 7s 116ms/step - loss: 0.8325 - accuracy: 0.5350 - val_loss: 0.6234 - val_accuracy: 0.6350
Epoch 2/30
50/50 [==============================] - 5s 94ms/step - loss: 0.6738 - accuracy: 0.6587 - val_loss: 0.5001 - val_accuracy: 0.7600
Epoch 3/30
50/50 [==============================] - 5s 97ms/step - loss: 0.5725 - accuracy: 0.7150 - val_loss: 0.4250 - val_accuracy: 0.8350
Epoch 4/30
50/50 [==============================] - 5s 92ms/step - loss: 0.4991 - accuracy: 0.7688 - val_loss: 0.3722 - val_accuracy: 0.8750
Epoch 5/30
50/50 [==============================] - 5s 95ms/step - loss: 0.4542 - accuracy: 0.7975 - val_loss: 0.3405 - val_accuracy: 0.9050
Epoch 6/30
50/50 [==============================] - 4s 90ms/step - loss: 0.4262 - accuracy: 0.8188 - val_loss: 0.3414 - val_accuracy: 0.8700
Epoch 7/30
50/50 [==============================] - 5s 91ms/step - loss

newwww

In [2]:
import os
import numpy as np
import cv2
from tensorflow.keras.utils import to_categorical

# === Folder Dataset ===
path_segar = r"C:\Users\Lenovo\Klasifikasi_ikan\datasets\normalized_output\segar"
path_tidak_segar = r"C:\Users\Lenovo\Klasifikasi_ikan\datasets\normalized_output\tidak_segar"

img_height, img_width = 224, 224

# === Fungsi untuk membaca dan resize citra ===
def load_images_from_folder(folder, label):
    images = []
    labels = []
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.resize(img, (img_width, img_height))
            img = img / 255.0  # Normalisasi [0–1]
            images.append(img)
            labels.append(label)
    return images, labels

# === Muat semua citra ===
images_segar, labels_segar = load_images_from_folder(path_segar, 0)  # Label 0 = Segar
images_tdk_segar, labels_tdk_segar = load_images_from_folder(path_tidak_segar, 1)  # Label 1 = Tidak Segar

# === Gabungkan data ===
X_data = np.array(images_segar + images_tdk_segar, dtype="float32")
y_data = np.array(labels_segar + labels_tdk_segar)

# === One-hot encoding label ===
y_data = to_categorical(y_data, num_classes=2)

print(f"✅ Dataset siap digunakan")
print(f"Total data: {len(X_data)}")
print(f"Shape X_data: {X_data.shape}")
print(f"Shape y_data: {y_data.shape}")

✅ Dataset siap digunakan
Total data: 1000
Shape X_data: (1000, 224, 224, 3)
Shape y_data: (1000, 2)


In [3]:
# === 1. Import Library ===
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# === 2. Parameter Dasar ===
img_height, img_width = 224, 224
num_classes = 2
k_folds = 5
epochs = 30
batch_size = 16
learning_rate = 0.0001

# === 3. Setup Folder Hasil ===
base_dir = r"C:\Users\Lenovo\Klasifikasi_ikan\hasil_model"

folders = [
    os.path.join(base_dir, "base", "models"),
    os.path.join(base_dir, "base", "history"),
    os.path.join(base_dir, "base", "plots"),
    os.path.join(base_dir, "base", "evaluation"),
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("[INFO] Struktur folder hasil_model berhasil dibuat!")

# === 4. Dataset ===
# Pastikan X_data dan y_data sudah ada sebelumnya
# Contoh: X_data.shape = (1000, 224, 224, 3), y_data.shape = (1000, 2)
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

acc_per_fold = []
loss_per_fold = []

# === 5. Training per Fold ===
fold_no = 1
for train_index, val_index in kf.split(X_data, y_data):
    print(f"\n==============================")
    print(f"🔹 Fold {fold_no}/{k_folds} sedang dilatih...")
    print(f"==============================")

    X_train, X_val = X_data[train_index], X_data[val_index]
    y_train, y_val = y_data[train_index], y_data[val_index]

    # === Model MobileNetV2 ===
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    predictions = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=predictions)

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # === Path untuk setiap hasil per fold ===
    fold_dir = os.path.join(base_dir, "base", "evaluation", f"fold_{fold_no}")
    os.makedirs(fold_dir, exist_ok=True)

    checkpoint_path = os.path.join(base_dir, "base", "models", f"model_fold_{fold_no}.keras")
    csv_path = os.path.join(base_dir, "base", "history", f"history_fold_{fold_no}.csv")
    plot_acc_path = os.path.join(base_dir, "base", "plots", f"accuracy_fold_{fold_no}.png")
    plot_loss_path = os.path.join(base_dir, "base", "plots", f"loss_fold_{fold_no}.png")
    eval_npz_path = os.path.join(fold_dir, f"model_evaluation_data_fold{fold_no}.npz")

    # === Callback ===
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ModelCheckpoint(filepath=checkpoint_path, save_best_only=True, monitor='val_accuracy', mode='max')
    ]

    # === Training ===
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1
    )

    # === Simpan history ke CSV ===
    pd.DataFrame(history.history).to_csv(csv_path, index=False)

    # === Plot Accuracy ===
    plt.figure(figsize=(8, 4))
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'Akurasi Fold {fold_no}')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.savefig(plot_acc_path)
    plt.close()

    # === Plot Loss ===
    plt.figure(figsize=(8, 4))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'Loss Fold {fold_no}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(plot_loss_path)
    plt.close()

    # === Evaluasi ===
    scores = model.evaluate(X_val, y_val, verbose=0)
    print(f"✅ Fold {fold_no} - Akurasi: {scores[1]*100:.2f}% | Loss: {scores[0]:.4f}")

    acc_per_fold.append(scores[1] * 100)
    loss_per_fold.append(scores[0])

    # === Simpan hasil prediksi untuk evaluasi Confusion Matrix nanti ===
    y_pred = model.predict(X_val)
    np.savez(eval_npz_path, y_true=y_val, y_pred=y_pred)
    print(f"📁 Data evaluasi fold-{fold_no} disimpan di: {eval_npz_path}")

    fold_no += 1

# === 6. Ringkasan Akhir ===
print("\n==============================")
print("📊 Hasil K-Fold Validation")
print("==============================")
for i in range(k_folds):
    print(f"Fold {i+1}: Akurasi = {acc_per_fold[i]:.2f}%, Loss = {loss_per_fold[i]:.4f}")

mean_acc = np.mean(acc_per_fold)
mean_loss = np.mean(loss_per_fold)
print(f"\n🔹 Rata-rata Akurasi: {mean_acc:.2f}%")
print(f"🔹 Rata-rata Loss: {mean_loss:.4f}")

# === 7. Simpan Ringkasan Umum ===
with open(os.path.join(base_dir, "base", "evaluation", "kfold_summary.txt"), "w") as f:
    for i in range(k_folds):
        f.write(f"Fold {i+1}: Akurasi = {acc_per_fold[i]:.2f}%, Loss = {loss_per_fold[i]:.4f}\n")
    f.write(f"\nRata-rata Akurasi: {mean_acc:.2f}%\n")
    f.write(f"Rata-rata Loss: {mean_loss:.4f}\n")

print("\n✅ Semua hasil training dan evaluasi tersimpan di folder:")
print(base_dir)


[INFO] Struktur folder hasil_model berhasil dibuat!

🔹 Fold 1/5 sedang dilatih...


Epoch 1/30


50/50 [==============================] - 7s 109ms/step - loss: 0.9042 - accuracy: 0.5100 - val_loss: 0.7640 - val_accuracy: 0.5600
Epoch 2/30
50/50 [==============================] - 5s 94ms/step - loss: 0.7306 - accuracy: 0.6075 - val_loss: 0.6008 - val_accuracy: 0.6450
Epoch 3/30
50/50 [==============================] - 5s 93ms/step - loss: 0.5858 - accuracy: 0.7225 - val_loss: 0.5097 - val_accuracy: 0.7150
Epoch 4/30
50/50 [==============================] - 5s 92ms/step - loss: 0.5373 - accuracy: 0.7425 - val_loss: 0.4389 - val_accuracy: 0.8000
Epoch 5/30
50/50 [==============================] - 4s 89ms/step - loss: 0.5045 - accuracy: 0.7663 - val_loss: 0.3902 - val_accuracy: 0.8300
Epoch 6/30
50/50 [==============================] - 4s 89ms/step - loss: 0.4410 - accuracy: 0.8037 - val_loss: 0.3516 - val_accuracy: 0.8750
Epoch 7/30
50/50 [==============================] - 4s 88ms/step - 